In [6]:
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever


In [60]:
stay_crawling = pd.read_csv('../../data/stay_crawling.csv')
sports_crawling = pd.read_csv('../../data/sports_crawling.csv')
df = pd.concat([stay_crawling, sports_crawling], ignore_index=True)

documents = []
 
# for _, row in df.iterrows():
#     # 값 없는 애들은 nan
#     content = f"""
#     장소명: {row['장소명']}
#     구분: {row['구분']}
#     카테고리: {row['카테고리']}
#     한줄설명: {row['한줄설명']}
#     리뷰: {row['리뷰']}
#     리뷰개수: {row['리뷰 개수']}
#     평점: {row['평점']}
#     태그: {row['태그']}
#     가격: {row['가격']}
#     영업시간: {row['영업시간']}
#     주소: {row['주소']}
#     연락처: {row['연락처']}
#     출처: {row['출처']}
#     """
    
#     doc = Document(page_content=content.strip()) # 공백제거 하나용..? strip()
#     documents.append(doc)

# (전체 루프 내부)
for _, row in df.iterrows():
    col_names = [
        ('장소명', '장소명'),
        ('구분', '구분'),
        ('설명', '설명'),
        ('메뉴', '메뉴'),
        ('카테고리', '카테고리'),
        ('한줄설명', '한줄설명'),
        ('리뷰', '리뷰'),
        ('리뷰개수', '리뷰 개수'),
        ('평점', '평점'),
        ('태그', '태그'),
        ('가격', '가격'),
        ('영업시간', '영업시간'),
        ('주소', '주소'),
        ('연락처', '연락처'),
        ('출처', '출처')
    ]
    
    valid_lines = []
    sort = ""
    
    for display_name, col_name in col_names:
        val = row.get(col_name)
        if pd.notna(val) and str(val).strip() != "":
            valid_lines.append(f"{display_name}: {val}")
            
        if display_name == '구분':
            sort = val
    
    content = "\n".join(valid_lines)
    
    if not content:
        continue
        
    doc = Document(page_content=content, metadata={"sort":sort})
    documents.append(doc)
    
    
# print(documents[100].page_content)
print(documents)
print(len(documents))

[Document(metadata={'sort': '숙박업체'}, page_content="장소명: 롯데호텔 부산\n구분: 숙박업체\n카테고리: 호텔\n리뷰: ['롯데호텔 알바한사람입니다  저도 일을 잘 한 편은 아니지만 , 정말 잘못된 체계라고 생각합니다 . 정직원을 뽑으시던지 계약직을 쓰시던지 알바를 시키면서 계약직을 원하시면 어떻게하자는건지 모르겠습니다 일을 알려주지도 않고 알아서 다 하기를 바라는 신기한 상황을 처음보아 후기를 남깁니다.  알바생을 진짜 직원인것처럼 부려 먹는 신기한 연회장,호텔입니다  ', '연회했는데 직원들이 센스가 하나도 없어서, 쓰레기통 가져다달라, 뜨거운물 가져다달라, 하나하나 말을 해야 겨우 챙겨주고, 서비스는 별1개도 아깝습니다. ', '서면 중심부에 있어서 교통도 편리하고 근처에 맛집도 많아서 좋아요. 호텔은 오래되었지만 관리가 잘되어 있어서 괜찮았어요. 저녁에 실외 수영장도 라이트업 되어서 예뻤고 사진찍기 좋아요. 실내 수영장은 수영하기 좋구요. 라운지 음식도 맛있고 라세느 조식도 맛있어요. 또, 직원분들이 진짜 친절하셔서 좋았어요.    ']\n리뷰개수: 116개\n평점: 4.3\n태그: ['#2인실', '#가족실', '#물놀이', '#비즈니스호텔', '#사우나', '#실외수영장', '#연회장', '#이동약자접근', '#조식서비스', '#호캉스']\n가격: {'룸타입슈페리어 더블 (Superior Double)': '205,700원', '룸타입슈페리어 트윈 (Superior twin)': '205,700원', '룸타입디럭스 더블 (Deluxe Double)': '229,900원', '룸타입디럭스 트윈 (Deluxe Twin)': '229,900원'}\n영업시간: 24시간 영업\n주소: 부산 부산진구 가야대로 772\n연락처: 051-810-1000\n출처: 카카오맵(2025)"), Document(metadata={'sort': '숙박업체'}, page_content="장소명: 시그니엘 부산\n구분: 숙박업체\n카테고리

In [61]:
# 지석님
json_df = pd.read_json('../../data/busan_rag_data.json')

# for _, row in json_df.iterrows():
#     # 값 없는 애들은 nan
#     content = f"""
#     장소명: {row['장소명']}
#     구분: {row['구분']}
#     설명: {row['설명']}
#     메뉴: {row['메뉴']}
#     영업시간: {row['영업시간']}
#     주소: {row['주소']}
#     연락처: {row['연락처']}
#     출처: {row['출처']}
#     """

#     doc = Document(page_content=content.strip())
#     documents.append(doc)


for _, row in json_df.iterrows():
    target_columns = [
    '장소명', '구분', '설명', '메뉴', '카테고리',
    '한줄설명', '리뷰', '리뷰 개수', '평점', '태그',
    '가격', '영업시간', '주소', '연락처', '출처'
    ]
    
    valid_lines = []
    sort = ""
    
    for col_name in target_columns:
        val = row.get(col_name)
        if pd.isna(val):
            continue
        if isinstance(val, (list, dict)) and len(val) == 0:
            continue
        if str(val).strip() == "":
            continue
        
        valid_lines.append(f"{col_name}: {val}")
        if col_name == '구분':
            sort = val
    
    content = "\n".join(valid_lines)
    
    if not content:
        continue
        
    doc = Document(page_content=content, metadata={"sort":sort})
    documents.append(doc)
    

print(documents[300])

print(len(documents))


page_content='장소명: 굿모닝홍콩
구분: 음식점
설명: 누구나 부담없이 즐길 수 있는 홍콩 음식
주소: 서전로47번길 19 1층
출처: 부산한골목-전포공구길' metadata={'sort': '음식점'}
1065


In [25]:
# openAI key
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

In [35]:
# embedding..
# 모델 선정 이유... 그냥 써보고 싶어서..
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2") # 잘 안뽑아지는뎅..
# embeddings = HuggingFaceEmbeddings(model="intfloat/multilingual-e5-small") # 임베딩 크기 384


In [ ]:
vectordb = Chroma(
    embedding_function=embeddings, # 아까 만든 HuggingFaceEmbeddings 객체
    collection_name="test",
    persist_directory="./vectorDB/chroma_db"
)

# 3. 배치 저장 (나눠서 저장)
batch_size = 100
total_docs = len(documents)

print(f"총 {total_docs}개 문서 저장 시작...")

for i in range(0, total_docs, batch_size):
    batch = documents[i : i + batch_size]
    vectordb.add_documents(batch)
    print(f"진행률: {min(i + batch_size, total_docs)} / {total_docs}")



총 1065개 문서 저장 시작...
진행률: 100 / 1065
진행률: 200 / 1065
진행률: 300 / 1065
진행률: 400 / 1065
진행률: 500 / 1065
진행률: 600 / 1065
진행률: 700 / 1065
진행률: 800 / 1065
진행률: 900 / 1065
진행률: 1000 / 1065
진행률: 1065 / 1065
✅ 저장 완료! 에러가 사라졌을 겁니다.


In [ ]:
# 문서에서 벡터 비디 생성
# vectordb = Chroma.from_documents( # 문서리스트로부터 벡터 저장소 생성
#     documents=documents, # 문서 리스트
#     embedding=embeddings, # 임베딩 함수
#     collection_name="test", # 생성할 컬렉션 이름
#     persist_directory="./vectorDB/chroma_db", # 컬렉션 저장할 디렉토리 -> 지정하지 않으면 메모리에 임시 저장
#     # ids: None, # 문서 ID 리스트 지정안하면 UUID
#     )

BadRequestError: Error code: 400 - {'error': {'message': 'Requested 339204 tokens, max 300000 tokens per request', 'type': 'max_tokens_per_request', 'param': None, 'code': 'max_tokens_per_request'}}

In [71]:
# 만들어진 db 에 연결 (load)
persist_db = Chroma(
    persist_directory="./vectorDB/chroma_db",
    embedding_function=embeddings,
    collection_name="test", # 컬렉션 이름 같아야함
)

In [ ]:
# DB 에 값 추가
persist_db.add_texts(
    texts = ["해운대의 더러운 호텔로는 네이버호텔이 있습니다."],
    # metadatas = [{}]
    ids=["1"],
    embedding_function = embeddings
)

# persist_db.add_documents()


['1']

In [1]:
persist_db.get(["1"]) # id 로만 조회 가능 PK 같은건가봄?
# persist_db.get(limit=5)

NameError: name 'persist_db' is not defined

In [73]:
# lang chain 썼을때...
query = "더러운 해운대의 호텔 추천해줘"
results = persist_db.similarity_search(query, k=3)

for res in results:
    print("-" * 30)
    print(res.page_content) 

------------------------------
장소명: 더반호텔 해운대점
구분: 숙박업체
카테고리: 호텔
리뷰: ['단순한 호텔로 생각했는데 차로 오면 프론트 직원분이 오셔서 응대해주시고 너무 친절했어요 수건과 물 추가 제공에  ott도 다 되고 깨끗하면서 리클라이너 침대에 공기청정기 구비, 자동 커튼에 객실용 패드에서 유튜브 뮤직 등 최신식입니다 해운대 코앞이라 위치도 좋고 모든게 좋았습니다 ', '컨디션 전반적으로 좋고 무료커피 서비스 굿 ', '이 근방에서 제일 좋은 호텔인듯 함 직원분 친절하시고 방 상태도 굿 ']
리뷰개수: 38개
평점: 4.6
태그: ['#2인실', '#가족호텔', '#비즈니스룸', '#비즈니스호텔', '#오션뷰', '#조식서비스', '#호캉스']
가격: {}
영업시간: [{'day': '화', 'hours': '15:00 ~ 12:00'}, {'day': '수', 'hours': '15:00 ~ 12:00'}, {'day': '목', 'hours': '15:00 ~ 12:00'}, {'day': '금', 'hours': '15:00 ~ 12:00'}, {'day': '토', 'hours': '15:00 ~ 12:00'}, {'day': '일', 'hours': '15:00 ~ 12:00'}, {'day': '월', 'hours': '15:00 ~ 12:00'}]
주소: 부산 해운대구 해운대해변로197번길 10-18
연락처: 0507-1376-3310
출처: 카카오맵(2025)
------------------------------
장소명: 해운대센텀호텔
구분: 숙박업체
카테고리: 호텔
리뷰: ['가격이나 객실 구조는 정말 좋은데, 청소 상태가 정말 아쉽습니다. 구석들을 많이 놓쳐서 온수 틀면 묵은 악취가 올라옵니다. 복도도 벽과 카펫에 얼룩이 많이 졌습니다. 높은 층들은 수도와 변기 수압이 낮습니다. ', '10041007 2.0과 3.0사이 어딘가. 펠릭스바이STX를 뒤로하고 잡은 객실. 악취 같은 것은 없었으며 예약한 

In [74]:
import pickle
from langchain_community.retrievers import BM25Retriever

BM25_PATH = "./vectorDB/bm25_index.pkl"

def build_bm25(documents, k=5):
    bm25 = BM25Retriever.from_documents(documents)
    bm25.k = k

    with open(BM25_PATH, "wb") as f:
        pickle.dump(bm25, f)

    return bm25


def load_bm25(k=5):
    with open(BM25_PATH, "rb") as f:
        bm25 = pickle.load(f)

    bm25.k = k
    return bm25


In [75]:
bm25_retriever = build_bm25(documents)


In [76]:
bm25_retriever = load_bm25(k=5)


In [77]:
# retriever = persist_db.as_retriever( # retriever 
#     search_type="similarity", # 유사도 검색
#     search_kwargs={"k": 3}    # 3 개 뽑기
# )

SEARCH_TYPE = [
    { # mmr 전용
    "k": 3, 
    "fetch_k": 10, 
    "lambda_mult": 0.5 # 다양성 조절 -> 0 으로 갈수록 다양성 up 
    },
    { 
    "k": 5,
    "score_threshold": 0.8 # 유사도 점수가 기준 이상인 애들만
    },
    {
    "k": 5,
    "distance_threshold": 0.2 # 임베딩 거리 기준
    },
    {
    "k": 3,
    "filter": {"source": "카카오맵(2025)"} # 메타 데이터 필터링!
    },]

# 상관없는 문서 없애기
# retriever = persist_db.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs=SEARCH_TYPE[1]
# )

# mmr(유사도 + 다양성) 전용값들
# dense
retriever = persist_db.as_retriever(
    search_type="mmr",
    search_kwargs= SEARCH_TYPE[0]
)

# dense
dense_retriever = persist_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# sparse
bm25_retriever = load_bm25(k=5)

# sparse + dense
ensemble_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.6, 0.4]
)


llm = ChatOpenAI(
    model="gpt-4o", 
    temperature=0
    )

template = """
당신은 장소 추천 전문가로서 여행 계획 플래너가 여행 계획을 세울 때 참고할 수 있는 숙박 업소를 추출하는 역할을 맡았습니다.
아래 [정보]를 바탕으로 질문에 친절하게 답해주세요. 안 좋은 정보라도 뽑아서 답하세요.
[정보]에 없는 내용은 지어내지 말고 "정보가 없습니다"라고 답하세요.

[정보]:
{context}

질문: {question}
"""
prompt = ChatPromptTemplate.from_template(template) # langchain 용 프롬프트 객체

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs]) # document -> str

rag_chain = (
    { # 입력을 두갈래로
        "context": ensemble_retriever | format_docs, # -> 리트리버가 질문 받고, 벡터 디비에서 문서 검색-> 결과 전달 format docs 가 문자열로 변환 
        "question": RunnablePassthrough() # 아무 처리 x 쿼리 그대로 전달
    }
    | prompt # 프롬프트 객체에 값 전달 
    | llm # 모델 호출
    | StrOutputParser() # 출력값 파싱 
)

query = "더러운 해운대 호텔 3개 추천해줄 수 있어? 참고해서 거기는 예약안하게"
response = rag_chain.invoke(query)

print(f"질문: {query}")
print(f"답변:\n{response}")


질문: 더러운 해운대 호텔 3개 추천해줄 수 있어? 참고해서 거기는 예약안하게
답변:
다음은 리뷰에서 청결 상태에 대한 부정적인 피드백이 있는 해운대 호텔 3곳입니다:

1. **해운대센텀호텔**: 리뷰에서 청소 상태가 아쉽다는 의견이 있으며, 구석을 놓쳐서 온수를 틀면 악취가 올라온다는 지적이 있습니다. 또한, 복도 벽과 카펫에 얼룩이 많다는 언급이 있습니다.

2. **해운대씨클라우드호텔레지던스**: 숙소 위생과 방음이 아쉽다는 리뷰가 있으며, 침구에 먼지가 많아 재채기가 계속 나왔다는 의견이 있습니다. 바닥도 미끌거리는 느낌이 들었다고 합니다.

3. **베니키아 프리미어 호텔 해운대**: 리뷰에서 옷장과 이불에 먼지가 많다는 지적이 있으며, 이불을 들추니 머리카락이 발견되었다는 의견이 있습니다.

이 호텔들은 청결 상태에 대한 부정적인 리뷰가 있으니 참고하시기 바랍니다.
